# Transformer Architecture

The transformer (Vaswani et al., 2017 — *Attention Is All You Need*) replaces recurrence and convolution entirely with **self-attention**: a mechanism that computes pairwise interactions between all tokens in a single parallelizable operation. Every pair of positions is $O(1)$ steps apart, eliminating the vanishing-gradient horizon that limited RNNs on long sequences.

## Scaled Dot-Product Attention

**Definition:** Given queries $\mathbf{Q} \in \mathbb{R}^{n \times d_k}$, keys $\mathbf{K} \in \mathbb{R}^{m \times d_k}$, and values $\mathbf{V} \in \mathbb{R}^{m \times d_v}$, the attention output is:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

**Step-by-step:**

1. **Similarity scores** $\mathbf{S} = \mathbf{Q}\mathbf{K}^\top \in \mathbb{R}^{n \times m}$: entry $S_{ij}$ measures how much query $i$ attends to key $j$.
2. **Scaling** by $1/\sqrt{d_k}$: without this, dot products grow in magnitude as $d_k$ increases, pushing softmax into near-zero-gradient saturation regions.
3. **Softmax** over keys: $A_{ij} = \dfrac{\exp(S_{ij}/\sqrt{d_k})}{\sum_{j'} \exp(S_{ij'}/\sqrt{d_k})}$, so each row of $\mathbf{A}$ sums to 1.
4. **Weighted sum** of values: $\text{Output} = \mathbf{A}\mathbf{V} \in \mathbb{R}^{n \times d_v}$.

**Soft dictionary intuition:** Keys and values form a differentiable lookup table. The query asks *what do I need?*, keys answer *what do I contain?*, and values are the associated content retrieved.

**Complexity:** $O(n^2 d_k)$ time and $O(n^2)$ memory — the quadratic bottleneck in sequence length $n$.

## Multi-Head Attention (MHA)

A single attention head captures one type of relationship. Multi-head attention runs $h$ independent heads in parallel with separate learned projections, then combines:

$$\text{MHA}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,\mathbf{W}^O$$

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q,\; \mathbf{K}\mathbf{W}_i^K,\; \mathbf{V}\mathbf{W}_i^V)$$

where $\mathbf{W}_i^Q, \mathbf{W}_i^K \in \mathbb{R}^{d_\text{model} \times d_k}$, $\mathbf{W}_i^V \in \mathbb{R}^{d_\text{model} \times d_v}$, $\mathbf{W}^O \in \mathbb{R}^{h d_v \times d_\text{model}}$.

Setting $d_k = d_v = d_\text{model}/h$ keeps total compute equal to single-head attention at full dimension. Different heads empirically specialize: some track syntactic dependencies, others coreference, positional proximity, or semantic roles.

### Parameter count

$$\underbrace{3 \cdot d_\text{model} \cdot d_\text{model}}_{\mathbf{W}^Q, \mathbf{W}^K, \mathbf{W}^V} + \underbrace{d_\text{model}^2}_{\mathbf{W}^O} = 4\,d_\text{model}^2$$

Independent of the number of heads $h$ (for fixed $d_\text{model}$).

## Multi-Query and Grouped-Query Attention

Standard MHA maintains $h$ separate KV heads, one per query head. At inference, the **KV cache** stores all past keys and values — its size is $O(n \cdot h \cdot d_k)$ per layer, which becomes a memory bottleneck for long contexts.

**Multi-Query Attention (MQA):** All query heads share a single KV head:

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q,\; \mathbf{K}\mathbf{W}^K,\; \mathbf{V}\mathbf{W}^V)$$

KV cache shrinks by a factor of $h$. Quality degrades slightly.

**Grouped-Query Attention (GQA):** Partition the $h$ query heads into $g$ groups; each group shares one KV head. MHA is the special case $g = h$; MQA is $g = 1$. GQA at $g = h/4$ recovers most of MHA quality at a fraction of the KV memory cost.

$$\text{KV cache size} = 2 \cdot n \cdot g \cdot d_k \cdot L \cdot \text{bytes}$$

where $L$ is the number of layers and $n$ is the sequence length. GQA is now standard in LLaMA 3, Mistral, and Gemma.

## Positional Encodings

Self-attention is **permutation equivariant**: shuffling input tokens shuffles outputs identically. Without positional information, the model cannot distinguish *the cat sat on the mat* from any permutation of those tokens. Positional encodings inject order into token representations.

### Sinusoidal (Vaswani et al., 2017)

Add a fixed encoding $\mathbf{PE}_{pos} \in \mathbb{R}^{d_\text{model}}$ to each token embedding:

$$\text{PE}_{(pos,\, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right), \qquad \text{PE}_{(pos,\, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_\text{model}}}\right)$$

Each dimension oscillates at a unique frequency. Key property: $\text{PE}_{pos+k}$ is a linear function of $\text{PE}_{pos}$ for any fixed offset $k$, allowing the model to represent relative positions via learned linear combinations.

### Learned Positional Embeddings

Train a lookup table $\mathbf{E} \in \mathbb{R}^{T_{\max} \times d_\text{model}}$. Flexible but cannot generalize beyond the maximum training length $T_{\max}$.

### Rotary Position Embedding (RoPE)

RoPE encodes position by rotating query and key vectors in 2D subspaces. For a pair of dimensions $(2i, 2i+1)$ at position $m$, the rotation matrix is:

$$\mathbf{R}_m^{(i)} = \begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix}, \qquad \theta_i = 10000^{-2i/d_k}$$

The full rotation applies these 2D rotations block-diagonally across all dimension pairs. The critical property: the inner product between a query at position $m$ and a key at position $n$ depends only on the **relative position** $m - n$:

$$\langle \mathbf{R}_m \mathbf{q},\; \mathbf{R}_n \mathbf{k} \rangle = \langle \mathbf{q},\; \mathbf{R}_{n-m} \mathbf{k} \rangle$$

This gives exact relative position information without requiring separate relative bias terms. RoPE generalizes better to longer sequences than absolute encodings and is used in LLaMA, Mistral, Gemma, and most modern LLMs.

### ALiBi (Press et al., 2022)

Add a fixed, non-learned negative bias proportional to distance to attention logits before softmax:

$$S_{ij} \leftarrow S_{ij} - m_h \cdot |i - j|$$

where $m_h$ is a head-specific slope. No position vectors at all. Generalizes gracefully beyond the training context window (extrapolation) and has no parameters.

## The Transformer Block

Each transformer layer consists of two sub-layers wrapped with residual connections:

$$\mathbf{h}' = \mathbf{h} + \text{MHA}(\text{Norm}(\mathbf{h}))$$
$$\mathbf{h}'' = \mathbf{h}' + \text{FFN}(\text{Norm}(\mathbf{h}'))$$

This is the **pre-norm** variant; the original transformer placed normalization after the residual add (post-norm). Pre-norm stabilizes training at depth and is now universal in large models.

### Feed-Forward Network (FFN)

Applied identically and independently to each token position:

$$\text{FFN}(\mathbf{x}) = \phi(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

with $\mathbf{W}_1 \in \mathbb{R}^{d_\text{model} \times d_{ff}}$, $\mathbf{W}_2 \in \mathbb{R}^{d_{ff} \times d_\text{model}}$, and $d_{ff} = 4\,d_\text{model}$ typically. The FFN expands into a higher-dimensional space, applies a pointwise nonlinearity, and projects back. More than half of transformer parameters reside here.

### SwiGLU

Modern LLMs replace the ReLU/GELU FFN with a **gated** variant:

$$\text{SwiGLU}(\mathbf{x}) = (\mathbf{x}\mathbf{W}_1 \otimes \text{Swish}(\mathbf{x}\mathbf{W}_g))\,\mathbf{W}_2$$

where $\text{Swish}(x) = x \cdot \sigma(x)$ and $\otimes$ is elementwise product. Three weight matrices instead of two; $d_{ff}$ is typically reduced to $\frac{2}{3} \cdot 4\,d_\text{model}$ to keep parameter count equivalent. Empirically outperforms both ReLU and GeLU.

### Layer Normalization

Normalizes over the **feature** dimension for each token independently (unlike BatchNorm which normalizes over the batch):

$$\text{LN}(\mathbf{x}) = \gamma \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta, \qquad \mu = \frac{1}{d}\sum_i x_i, \quad \sigma^2 = \frac{1}{d}\sum_i (x_i - \mu)^2$$

where $\gamma, \beta \in \mathbb{R}^d$ are learned scale and shift parameters. Independent of batch size — essential for variable-length sequences and single-example inference.

### RMSNorm

Drops the mean-centering step from LayerNorm:

$$\text{RMSNorm}(\mathbf{x}) = \gamma \frac{\mathbf{x}}{\text{RMS}(\mathbf{x})}, \qquad \text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}$$

Cheaper to compute, comparable in quality, and now preferred over LayerNorm in LLaMA, Mistral, and Gemma.

### Residual Connections

The residual stream $\mathbf{h}^{(\ell)} = \mathbf{h}^{(\ell-1)} + F^{(\ell)}(\mathbf{h}^{(\ell-1)})$ ensures a gradient highway from the loss to early layers:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(0)}} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}^{(L)}} \prod_{\ell=1}^{L} \left(\mathbf{I} + \frac{\partial F^{(\ell)}}{\partial \mathbf{h}^{(\ell-1)}}\right)$$

The identity term prevents the product from collapsing to zero regardless of depth. Each layer learns a *residual correction* on top of the identity map.

## Masking

**Causal (autoregressive) mask:** For decoder-only generation, token $i$ must not attend to token $j > i$ (future tokens are unavailable). Apply an upper-triangular $-\infty$ mask to scores before softmax:

$$\tilde{S}_{ij} = \begin{cases} S_{ij} & j \le i \\ -\infty & j > i \end{cases}$$

After softmax, $-\infty$ scores yield attention weight 0, so no information flows from future positions.

**Padding mask:** Sequences in a batch are right-padded to uniform length. Padding tokens should not influence representations — mask them with $-\infty$ at the key dimension before softmax.

**Prefix mask:** In prefix language modeling (used in T5, Gemini), the prompt tokens attend bidirectionally to one another, and generation tokens attend causally. The mask is block-structured: fully visible upper-left block (prefix×prefix), causal lower-right block (generation).

## Encoder Architecture

An encoder maps an input sequence to a sequence of contextual representations. It uses **bidirectional self-attention**: every token attends to all others, so each output representation integrates global context.

$$\mathbf{H}^{(0)} = \text{TokenEmbed}(\mathbf{x}) + \mathbf{PE}$$
$$\mathbf{H}^{(\ell)} = \text{TransformerBlock}^{(\ell)}(\mathbf{H}^{(\ell-1)}), \quad \ell = 1, \ldots, L$$

**BERT** (Devlin et al., 2018) is the canonical encoder-only model. It prepends a special `[CLS]` token whose final-layer representation serves as an aggregate sequence embedding for classification tasks.

### Masked Language Modeling (MLM)

Pre-training objective: randomly replace 15% of tokens with `[MASK]`, 10% with a random token, 10% with the original, and train the model to predict the original token from context:

$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in \mathcal{M}} \log P(x_i \mid \mathbf{x}_{\setminus \mathcal{M}})$$

where $\mathcal{M}$ is the set of masked positions. MLM forces bidirectional contextual understanding but cannot be used directly for autoregressive generation.

### Next Sentence Prediction (NSP)

Used in original BERT: predict whether sentence B follows sentence A. Later found to be mostly unhelpful (RoBERTa showed dropping NSP improves downstream tasks). Subsequent work uses span masking or whole-word masking instead.

## Decoder Architecture

A decoder generates tokens autoregressively under a causal mask. Each token attends only to itself and earlier tokens.

$$P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

### Causal Language Modeling (CLM)

$$\mathcal{L}_{\text{CLM}} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t \mid x_{<t}; \theta)$$

The simplest pre-training objective — predict the next token. Scales naturally: more data and parameters reliably lower loss. The dominant paradigm for all large language models (GPT-2/3/4, LLaMA, Mistral, Claude).

### KV Cache

At each generation step $t$, the keys and values for all previous positions $1, \ldots, t-1$ are identical to those computed at step $t-1$. The **KV cache** stores these tensors and reuses them, reducing per-step compute from $O(t^2)$ to $O(t)$:

- **Prefill phase:** Process the entire prompt in parallel, populate the KV cache.
- **Decode phase:** At each step, compute $\mathbf{q}_t$ for the new token only, attend to all cached $\mathbf{K}, \mathbf{V}$, append the new $\mathbf{k}_t, \mathbf{v}_t$.

KV cache memory: $2 \times n \times L \times h \times d_k \times \text{bytes}$. For LLaMA-3 70B at context length 8192: ~35 GB in fp16 with full MHA; GQA reduces this ~4–8$\times$.

## Encoder-Decoder Architecture

For sequence-to-sequence tasks (translation, summarization), the encoder processes the full source sequence bidirectionally and the decoder generates the target autoregressively.

The decoder has **three sub-layers** per block:

1. **Causal self-attention** over decoder outputs so far.
2. **Cross-attention** from decoder to encoder:
   $$\text{CrossAttn}(\mathbf{H}^\text{dec}, \mathbf{H}^\text{enc}) = \text{Attention}(\mathbf{H}^\text{dec}\mathbf{W}^Q,\; \mathbf{H}^\text{enc}\mathbf{W}^K,\; \mathbf{H}^\text{enc}\mathbf{W}^V)$$
   Queries come from the decoder; keys and values come from the encoder. This is the mechanization of Bahdanau attention.
3. **FFN** applied positionwise.

Each sub-layer is wrapped with a residual connection and normalization.

**T5** (Raffel et al., 2020) casts all NLP tasks as text-to-text (classification → generating the class name, summarization → generating the summary) and trains an encoder-decoder with span masking. **BART** (Lewis et al., 2020) adds a denoising pre-training objective: corrupt the input (masking, deletion, permutation) and train the decoder to reconstruct the original.

## Efficient Attention

The $O(n^2)$ cost of full attention limits context length. Various approaches trade off exactness, approximation quality, and hardware efficiency.

| Method | Time | Memory | Key idea |
|---|---|---|---|
| Full attention | $O(n^2 d)$ | $O(n^2)$ | Exact |
| Sparse attention | $O(n\sqrt{n}\,d)$ | $O(n\sqrt{n})$ | Local + strided patterns |
| Linformer | $O(nd)$ | $O(nd)$ | Low-rank projection of keys/values to length $k \ll n$ |
| Performer | $O(nd)$ | $O(nd)$ | Random feature approximation of softmax kernel |
| FlashAttention | $O(n^2 d)$ | $O(n)$ | IO-aware tiling; same FLOPs, 2–4$\times$ wall-clock speedup |
| Ring attention | $O(n^2 d / P)$ | $O(n/P)$ | Distribute attention across $P$ devices |

### FlashAttention (Dao et al., 2022)

Full attention materializes the $n \times n$ attention matrix in GPU HBM (high-bandwidth memory), which is the bottleneck — not FLOPs. FlashAttention fuses the softmax, matmul, and dropout into a single CUDA kernel that tiles the computation in SRAM (on-chip cache):

1. Partition $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ into blocks that fit in SRAM.
2. For each query block, iterate over key-value blocks, accumulate the softmax-weighted sum using the **online softmax** trick (update running max and normalization constant):
   $$m_i^{\text{new}} = \max(m_i^{\text{old}}, \max_j S_{ij}), \qquad \ell_i^{\text{new}} = e^{m_i^{\text{old}} - m_i^{\text{new}}} \ell_i^{\text{old}} + \sum_j e^{S_{ij} - m_i^{\text{new}}}$$
3. Never write the full $n \times n$ matrix to HBM.

**Memory:** $O(n)$ instead of $O(n^2)$. Backward pass recomputes attention scores from $\mathbf{Q}, \mathbf{K}$ rather than storing them (gradient checkpointing at the attention level).

## Training Objectives

### Causal Language Modeling (CLM)

$$\mathcal{L}_{\text{CLM}}(\theta) = -\mathbb{E}_{x \sim \mathcal{D}}\left[\frac{1}{T}\sum_{t=1}^{T} \log P_\theta(x_t \mid x_{<t})\right]$$

Standard autoregressive next-token prediction. Loss equals cross-entropy. Perplexity is the exponentiated average loss: $\text{PPL} = e^{\mathcal{L}}$.

### Masked Language Modeling (MLM)

$$\mathcal{L}_{\text{MLM}}(\theta) = -\mathbb{E}_{x \sim \mathcal{D}}\left[\sum_{i \in \mathcal{M}} \log P_\theta(x_i \mid \mathbf{x}_{\setminus \mathcal{M}})\right]$$

Predicts masked tokens from full (bidirectional) context. Computationally less efficient than CLM: only 15% of tokens contribute to the loss per forward pass.

### Span Masking (T5, SpanBERT)

Rather than masking individual tokens, mask contiguous spans of mean length $\ell$ (T5 uses $\ell = 3$). Harder than token-level masking; forces the model to generate multi-token sequences, which correlates better with downstream generation tasks.

### RLHF and RLAIF

After supervised pre-training, align the model with human preferences:

1. **Supervised Fine-Tuning (SFT):** Fine-tune on a small set of human-written demonstrations.
2. **Reward Model (RM):** Train a Bradley-Terry model on human preference comparisons $(y_w \succ y_l \mid x)$:
   $$\mathcal{L}_{\text{RM}} = -\mathbb{E}[\log \sigma(r(x, y_w) - r(x, y_l))]$$
3. **PPO fine-tuning:** Maximize expected reward while penalizing KL divergence from the SFT policy:
   $$\mathcal{L}_{\text{PPO}} = \mathbb{E}[r(x, y)] - \beta\, \text{KL}(\pi_\theta \| \pi_{\text{SFT}})$$

**Direct Preference Optimization (DPO)** (Rafailov et al., 2023) eliminates the explicit RM by deriving a closed-form supervised objective from the PPO optimality conditions:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\!\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{SFT}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{SFT}}(y_l|x)}\right)\right]$$

## Scaling Laws

### Kaplan et al. (2020)

Language model loss follows predictable power laws in model parameters $N$, dataset size $D$, and compute $C \approx 6ND$:

$$\mathcal{L}(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N}, \qquad \mathcal{L}(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D}$$

with empirical exponents $\alpha_N \approx 0.076$, $\alpha_D \approx 0.095$. Larger models are more sample-efficient: a 10$\times$ larger model needs roughly 5.5$\times$ fewer tokens to reach the same loss.

### Chinchilla (Hoffmann et al., 2022)

Kaplan et al. held $D$ roughly fixed and scaled $N$ — but this is suboptimal. For a fixed compute budget $C$, jointly optimize:

$$N^* = \arg\min_{N:\, 6ND = C} \mathcal{L}(N, D)$$

The Chinchilla result: optimal allocation uses roughly **20 tokens per parameter** — $N^* \approx D^* / 20$. Prior models (GPT-3, Gopher) were massively undertrained. A 70B model trained on 1.4T tokens (Chinchilla-optimal) outperforms a 280B model trained on 300B tokens (Gopher) at equal compute.

**Practical implication:** At inference time, a smaller, well-trained model is cheaper to serve. The field now trains smaller models on more tokens — LLaMA 3 8B trained on 15T tokens significantly exceeds the Chinchilla-optimal for that model size.

## Vision Transformer (ViT)

Dosovitskiy et al. (2020) apply a standard transformer encoder directly to images by treating flattened patches as tokens.

**Patch embedding:** Divide an $H \times W \times C$ image into $N = \frac{HW}{P^2}$ non-overlapping $P \times P$ patches, flatten each to $\mathbb{R}^{P^2 C}$, and project to $\mathbb{R}^{d_\text{model}}$:

$$\mathbf{z}_0 = [\mathbf{x}_{\text{cls}};\; \mathbf{x}^1_p \mathbf{E};\; \ldots;\; \mathbf{x}^N_p \mathbf{E}] + \mathbf{E}_{\text{pos}}, \qquad \mathbf{E} \in \mathbb{R}^{P^2 C \times d_\text{model}}$$

Prepend a learnable `[CLS]` token; add learned positional embeddings.

**Encoder:** $L$ standard transformer encoder blocks over the $N+1$ tokens.

**Classification head:** $\mathbf{z}_L^0$ (the `[CLS]` token output) → MLP → logits.

**Inductive bias:** ViT has essentially no spatial inductive bias — it treats all pairs of patches equally until it learns otherwise from data. CNNs encode locality and translation equivariance by design. Consequently:
- ViT underperforms CNNs on small datasets (e.g., ImageNet-1K alone).
- ViT outperforms CNNs at very large scale (JFT-300M, ~3B image-label pairs).

**MAE** (He et al., 2022): mask 75% of patches and train the ViT encoder + lightweight decoder to reconstruct raw pixel values. High masking forces semantic understanding rather than local interpolation. The encoder alone (discarding the decoder) produces strong representations for downstream tasks.

**CLIP** (Radford et al., 2021): train paired image (ViT) and text (transformer) encoders contrastively so that matching image-text pairs have maximum cosine similarity:

$$\mathcal{L}_{\text{CLIP}} = -\frac{1}{N}\sum_{i=1}^N \log \frac{\exp(\mathbf{v}_i \cdot \mathbf{t}_i / \tau)}{\sum_{j=1}^N \exp(\mathbf{v}_i \cdot \mathbf{t}_j / \tau)}$$

Learns aligned multimodal representations without class labels. Enables zero-shot image classification.

## Modern LLM Design Summary

| Component | Original Transformer (2017) | Modern LLMs (e.g., LLaMA 3) |
|---|---|---|
| Normalization | Post-norm (LayerNorm) | Pre-norm (RMSNorm) |
| Positional encoding | Sinusoidal (absolute) | RoPE (relative) |
| Activation | ReLU | SwiGLU |
| Attention | Full MHA | Grouped Query Attention (GQA) |
| Architecture | Encoder-Decoder | Decoder-only |
| Context | 512 tokens | 8K–128K tokens |
| Training tokens | ~250K steps on WMT | 15T+ tokens |

**Parameter count** of a decoder-only transformer with $L$ layers, $d_\text{model}$ width, $d_{ff}$ FFN hidden dim, vocabulary size $V$:

$$N \approx \underbrace{V \cdot d_\text{model}}_{\text{embedding}} + L \cdot \left(\underbrace{4\,d_\text{model}^2}_{\text{MHA}} + \underbrace{2\,d_\text{model}\,d_{ff}}_{\text{FFN}} + \underbrace{\epsilon}_{\text{norms}}\right)$$

With $d_{ff} = 4\,d_\text{model}$: the FFN dominates at $8\,d_\text{model}^2$ per layer vs. $4\,d_\text{model}^2$ for attention.